# 02 — El primer bucle agéntico

**Bloque 2, 30 minutos.** El código de este bloque vive en
[`src/paso2_herramienta/agente.py`](../src/paso2_herramienta/agente.py), y ese
archivo es el que vas a editar. El notebook lo importa en lugar de copiarlo, así
que lo que corres aquí es exactamente lo que tienes abierto en el editor.

Un agente es un programa: tiene un bucle, un criterio de terminación y módulos
que se importan. Eso no se escribe en celdas sin aprender un modelo mental
equivocado, y por eso el archivo está aparte.

**Corre el notebook de arriba abajo.** Tarda alrededor de un minuto: 46
segundos, medidos el 2026-11-15 contra el servidor del Tec con `qwen3.5:9b`. Un
tercio se va en la celda que compara cuatro cuentas.

> **Para quien lo proyecta.** Con `importlib.reload`, lo que se ve en el
> proyector es el archivo que el grupo está editando, sin reiniciar el kernel.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")
import paso2_herramienta.agente as p2
importlib.reload(p2)   # volver a correr esto después de editar el archivo

## Lo que hay en el sistema, antes de preguntar

El "sistema" del paso 2 es [`datos/inventario.csv`](../datos/inventario.csv),
leído directo del disco con la librería `csv`. **Sin HTTP, sin Docker y sin
red:** eso llega en el paso 3.

Es el mismo archivo que usa el backend, así que el paso 2 y el paso 3 ven el
mismo inventario. Lo que cambia entre un paso y otro no son los datos, es cómo
se llega a ellos.

La celda siguiente imprime la tabla completa. Tenla a la vista **antes** de
preguntarle al modelo: si no sabes qué hay, no puedes distinguir una respuesta
correcta de una inventada, y distinguirlas es lo que enseña este bloque.

In [ ]:
# El inventario completo. El modelo NO ve esta tabla: solo ve el esquema de la
# tool, y el resultado que le devolvemos cuando la llama.
print(f"  {'SKU':<15} {'producto':<30} {'existencias':>11}  {'precio':>7}")
print(f"  {'-' * 15} {'-' * 30} {'-' * 11}  {'-' * 7}")
for sku, p in p2.INVENTARIO.items():
    aviso = "  <- en cero" if p["existencias"] == 0 else ("  <- quedan pocas"
                                                          if p["existencias"] < 5 else "")
    print(f"  {sku:<15} {p['nombre']:<30} {p['existencias']:>11}  {p['precio']:>7}{aviso}")

print(f"\n  {len(p2.INVENTARIO)} productos. Fíjate en los tres cafés: 'cuánto café hay'")
print("  no tiene una sola respuesta, y eso lo vamos a usar más abajo.")

## El esquema de una tool

La celda siguiente imprime el esquema JSON de las dos tools. Eso es lo único que
el modelo ve de tus funciones: **no ve el código.**

**La `description` es prompt, no documentación.** Es el lugar donde más rinde el
tiempo de ajuste, y el que casi nadie ajusta. El notebook 07 mide un caso en el
que cambiar una descripción decidió si el agente llamaba o no a una tool.

Lee con calma la de `consultar_catalogo`, porque de ahí sale todo lo que pasa
después: **declara lo que la tool NO devuelve.** El catálogo da clave, nombre y
precio. Las existencias no. Para saber cuántas unidades hay de algo, el modelo
tiene que pedir `consultar_stock` con una clave que primero tuvo que leer.

> **Cuánto devuelve una tool decide cuántas vueltas da el agente, y eso lo
> decides tú.** El comentario de `consultar_catalogo` en
> [`agente.py`](../src/paso2_herramienta/agente.py) tiene lo que se midió al
> tomar esta decisión: con las existencias dentro del catálogo, cualquier
> pregunta por cantidades se resolvía en una sola llamada.

In [ ]:
import json

# Las DOS, y en este orden: la del catalogo dice lo que NO devuelve, y de ahi
# sale todo lo que hace el agente en las celdas de abajo.
for t in p2.HERRAMIENTAS:
    print(json.dumps(t, indent=2, ensure_ascii=False))
    print()

## Una sola vuelta, a mano

Antes del bucle, una vuelta suelta. Fíjate en lo que devuelve la celda
siguiente: `tool_calls`, no una respuesta de texto. El modelo **pidió** que se
ejecutara una función; no ejecutó nada.

In [ ]:
from agentlib import chat, normalizar_argumentos

mensajes = [
    {"role": "system", "content": p2.SYSTEM},
    {"role": "user", "content": "¿Cuántas unidades hay de CAFE-500?"},
]
respuesta1 = chat(mensajes, herramientas=p2.HERRAMIENTAS)
print("respuesta1:", json.dumps(respuesta1, indent=2, ensure_ascii=False))
print("content   :", repr(respuesta1.get("content")))
print("tool_calls:", json.dumps(respuesta1.get("tool_calls"), indent=2, ensure_ascii=False))

La celda siguiente ejecuta la función y le devuelve el resultado al modelo con
el rol `tool`.

Ese es el punto mecánico del que cuelga todo lo demás: **el modelo nunca tocó el
sistema.** El diccionario `DISPONIBLES` es el puente, y es también el primer
guardrail, porque lo que no está ahí no corre. Nunca un `eval` sobre un nombre
que vino del modelo.

In [ ]:
tools = respuesta1["tool_calls"]
pedido = tools[0]
nombre = pedido["function"]["name"]
argumentos = normalizar_argumentos(pedido)
print(f"el modelo pidió: {nombre}({argumentos})")

resultado_tool = p2.DISPONIBLES[nombre](**argumentos)
print(f"ejecutamos y da: {resultado_tool}")


mensajes += [respuesta1, {"role": "tool", "content": json.dumps(resultado_tool, ensure_ascii=False)}]
respuesta2 = chat(mensajes, herramientas=p2.HERRAMIENTAS)
print("respuesta2:", json.dumps(respuesta2, indent=2, ensure_ascii=False))
print("\nrespuesta final:", respuesta2["content"])

## Y eso, en un `while`

La celda siguiente repite la vuelta a mano de la sección anterior hasta que el
modelo responde sin pedir nada. Son unas sesenta líneas, y debajo de cualquier
librería de agentes hay esto.

Después vienen cinco preguntas: una simple, una sobre un producto agotado, una
sobre un producto que no existe, una que no debería llamar ninguna tool, y una
ambigua, porque hay tres cafés.

**En cada una, cuenta las vueltas y cuántas tools pide por vuelta.** Como el
catálogo no trae existencias, hasta la pregunta más simple necesita dos tools
encadenadas: el catálogo para la clave, `consultar_stock` para el número. La
ambigua necesita cuatro, y pide tres de ellas en la misma vuelta.

In [ ]:
# El caso simple: un solo producto se llama "miel". Con el catalogo sin
# existencias, ni esto se resuelve en una llamada.
p2.correr("¿Cuántas unidades hay de miel?")

In [ ]:
# Existencias en cero. Mira si distingue 'existe' de 'hay'.
p2.correr("¿Hay té verde disponible?")

In [ ]:
# Un producto que no existe. Inventa, o avisa?
p2.correr("¿Cuántos tornillos tienes?")

In [ ]:
# Y lo que NO deberia llamar ninguna tool.
# Si aqui pide algo, ya tienes el bucle infinito del bloque 5 sin provocarlo.
p2.correr("Hola, ¿cómo estás?")

In [ ]:
# Y ahora la interesante: hay TRES cafes. Que hace el modelo?
# Pregunta cual? Elige uno y no avisa? Los suma? Lista los tres?
# No hay una respuesta correcta unica, y por eso sirve de ejemplo.
p2.correr("¿Cuánto café tenemos?")

## Dos preguntas que parecen la misma

Las dos celdas siguientes preguntan exactamente lo mismo. La única diferencia
son **los acentos en los nombres de los productos**.

Corre las dos y compara tres cosas: qué tool pide cada una primero, cuántas
vueltas da, y de dónde salió cada número de la respuesta final.

In [ ]:
# A — "café" y "té" CON acento
respuesta_a = p2.correr("¿Cuántas unidades tenemos de café y té?")

In [ ]:
# B — "cafe" y "te" SIN acento. Lo demas, identico.
respuesta_b = p2.correr("¿Cuantas unidades tenemos de cafe y te?")

### Qué pasó

| | primera vuelta | vueltas | tools | el número del té |
|---|---|---|---|---|
| **A** (`café`, `té`) | `consultar_stock` y `consultar_catalogo`, juntas | 3 | 4 | lo consultó: **0** |
| **B** (`cafe`, `te`) | `consultar_stock({'sku': 'CAFE-500'})` y `consultar_stock({'sku': 'TE-100'})` | 2 | 2 | no lo consultó |

**El acento cambia el camino que toma.** Con `café` y `té`, el modelo pide el
catálogo en la primera vuelta, saca de ahí las claves `TE-200` y `TE-MANZ-100`,
y en la segunda pide las existencias de las dos. Con `cafe` y `te`, adivina
claves: pidió `CAFE-500` y `TE-100`.

`TE-100` no existe. El error que le volvió traía **las diez claves válidas**, y
`TE-200` estaba entre ellas. B no las usó: cerró en la vuelta siguiente sin
pedir nada más.

**Y lo que contestó no coincide con el inventario en dos puntos.** Dijo *"la
clave correcta es TE-200 (Té negro 200g), que tenemos en stock"*. El catálogo lo
llama *Té verde 200g*, y ese producto tiene 0 unidades. La celda de abajo marca
las dos cosas buscándolas en el texto, así que no hay que creerle a este
párrafo.

> **Lo que B no hizo importa más que un número inventado.** No dio una cifra
> falsa: evitó tener que darla. Describió un producto que no consultó, con un
> nombre que no es el suyo, y afirmó que hay existencias de lo único que está
> agotado. Un sistema de pedidos leería eso como un sí.

**Que el acento decida la tool está medido.** El 2026-11-06, 5 de 5 corridas en
cada sentido, a temperatura 0, y aislando una variable a la vez: manda el acento
del *producto*, no el del adverbio, porque `cuantas` sin acento con `café`
acentuado va al catálogo. Está en
[`planeacion/hallazgos.md`](../planeacion/hallazgos.md).

> **Lo que ya no se reproduce es la invención de los cinco números.** Con una
> copia anterior de `qwen3.5:9b`, B pedía `consultar_stock({'sku': 'CAFE'})` y
> cerraba con cinco cantidades, ninguna real. Sigue firme qué tool elige cada
> variante; cambió qué tan mal termina. Las dos corridas están en `hallazgos.md`,
> con la fecha de cada una.

In [ ]:
# Confronta las DOS respuestas con el inventario buscando en su texto, no
# transcribiendolas a mano: si tu corrida contesta otra cosa, esto la sigue.
# Y las revisa con el mismo codigo: presuponer cual de las dos se equivoco es
# justo el error que hay que no cometer.
import re

UNIDADES = re.compile(r"(\d{1,5}) unidades")
SKU = re.compile(r"[A-Z]{2,}[A-Z0-9]*(?:-[A-Z0-9]+)+")
REALES = {v["existencias"] for v in p2.INVENTARIO.values()}


def revisar(etiqueta, texto):
    print(f"  {etiqueta}")
    for sku in dict.fromkeys(SKU.findall(texto)):
        p = p2.INVENTARIO.get(sku)
        if p is None:
            print(f"    {sku:<14} NO EXISTE en el catalogo")
            continue
        print(f"    {sku:<14} {p['nombre']:<24} {p['existencias']:>3} unidades reales")
        if p["nombre"].lower() not in texto.lower():
            print(f"    {'':<14} ^ la respuesta lo llama de otra forma")
    citadas = [int(n) for n in UNIDADES.findall(texto)]
    sin_respaldo = [n for n in citadas if n not in REALES]
    print(f"    cifras de unidades que da : {citadas or 'ninguna'}")
    print(f"    sin respaldo en inventario: {sin_respaldo or 'ninguna'}")


revisar("A  (café, té):", respuesta_a)
print()
revisar("B  (cafe, te):", respuesta_b)
print()
print("  Y el system prompt dice, literalmente:")
print(f"    ...{p2.SYSTEM[-58:]}")

### Lo que esto deja fijo

**El system prompt decía "no inventes números ni claves de producto".** Lo decía,
y B llamó al `TE-200` *"Té negro 200g"*. Un system prompt es una sugerencia
fuerte, no un control: una garantía va en tu código, y eso es el paso 4.

**Devolver el error como dato da la oportunidad de corregirse, no la impone.** B
recibió las diez claves válidas y cerró sin usarlas. El paso 3 tiene el caso
contrario, donde el modelo sí se corrige con la misma información. Las dos cosas
ocurren, y conviene enseñar las dos.

**Una respuesta equivocada se ve igual que una correcta.** *"La clave correcta
es TE-200, que tenemos en stock"* está bien escrito, es específico y suena
verificado. Sin la tabla del inventario al lado no hay con qué notarlo, y por eso
la primera celda del notebook la imprime.

**Y es determinista.** Dos corridas completas del notebook, el mismo día contra
el mismo servidor, dieron respuestas idénticas carácter por carácter. Con el
mismo código y el mismo contexto, si sale una vez sale siempre.

> **A tampoco contestó completo.** Reportó un café de los tres que hay, y ningún
> número suyo es inventado. Cuando se compara un caso bueno contra uno malo,
> conviene mirar también qué le falta al bueno.

> **Escribir sin acentos es normal cuando uno teclea rápido**, y aquí eso cambió
> la tool que el agente eligió. No es un detalle ortográfico: es que no se puede
> predecir qué tool va a elegir el modelo leyendo el prompt. Se mide.

### Cómo se arregla

No con un prompt más insistente. Las dos salidas son de diseño:

**Darle una tool que sirva para lo que preguntan.** El usuario buscó por nombre
y solo hay búsqueda por clave exacta. El `TODO 1` de
[`agente.py`](../src/paso2_herramienta/agente.py) propone `buscar_producto(texto)`;
mide si con esa tool el caso B deja de adivinar claves.

**O endurecer la descripción de `consultar_stock`**, para que diga que el `sku`
sale de `consultar_catalogo` y no se inventa. Edita el esquema, vuelve a correr
la celda del `importlib.reload` y mira si cambia. A veces alcanza y a veces no;
averiguar cuál de las dos es el ejercicio.

> La descripción de `consultar_catalogo` ya pasó por ese ajuste: dice lo que no
> devuelve, y eso es lo que hace que el agente encadene dos tools en vez de
> resolverlo con una.

## El agente no sabe hacer cuentas

La celda siguiente le pide un precio total. El modelo pide el catálogo, pide las
existencias de los dos productos y da un número, así que parece que calculó.

**No calculó.** No hay ninguna tool de calculadora todavía: el modelo *generó*
los dígitos del resultado prediciendo tokens, igual que genera palabras. Que
salga bien es otra cosa.

In [ ]:
p2.correr("¿Cuánto te tengo que pagar por 3 cafés de 1kg y 2 azúcares?")

# La cuenta real, en Python:
correcto = 3 * p2.INVENTARIO["CAFE-GRANO-1K"]["precio"] + 2 * p2.INVENTARIO["AZUCAR-1K"]["precio"]
print(f"\n  lo correcto es: {correcto:.0f}")

**Ese salió mal, y se delata en su propio desglose.** El modelo escribió
*"3 cafés de 1kg: $1,020 (3 x $340)"* y *"2 azúcares de 1kg: $76 (2 x $38)"*,
que son las dos multiplicaciones correctas, y cerró con un total de $1,560.
La suma de sus dos líneas es 1 096.

Aquí no hace falta el inventario para cacharlo. La respuesta se contradice sola,
dos líneas abajo del número equivocado, y eso es lo que significa que los
dígitos se generen en vez de calcularse.

> Fíjate también en que pidió las existencias de los dos productos aunque la
> pregunta era de precio, y el precio venía en el catálogo. Pedir de más también
> es una decisión del modelo, y también cuesta vueltas.

La celda siguiente compara cuatro cuentas contra Python. **Tarda unos 16
segundos**, medidos el 2026-11-15 contra el servidor del Tec.

In [ ]:
import contextlib, io, re

P = p2.INVENTARIO
CASOS = [
    ("¿Cuánto pago por 3 cafés de 1kg y 2 azúcares?",
     3*P["CAFE-GRANO-1K"]["precio"] + 2*P["AZUCAR-1K"]["precio"]),
    ("¿Cuánto pago por 7 cafés de 1kg, 13 tés de manzanilla y 11 vasos?",
     7*P["CAFE-GRANO-1K"]["precio"] + 13*P["TE-MANZ-100"]["precio"] + 11*P["VASO-360"]["precio"]),
    ("¿Cuánto cuesta llevarme TODO el inventario?",
     sum(v["existencias"]*v["precio"] for v in P.values())),
    ("Si compro 23 mieles y 19 leches deslactosadas, ¿cuánto pago?",
     23*P["MIEL-500"]["precio"] + 19*P["LECHE-DESL-1L"]["precio"]),
]

for pregunta, correcto in CASOS:
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        p2.correr(pregunta)
    final = buf.getvalue().split("responde:")[-1]
    # El regex NO puede ser r"\d[\d.]*": con "= $1,020.00." captura
    # '1020.00.' -y float() revienta con el punto final de la oracion-.
    dichos = {int(float(x)) for x in re.findall(r"\d+(?:\.\d+)?", final.replace(",", ""))
              if float(x) == int(float(x))}
    ok = int(correcto) in dichos
    print(f"  {'OK ' if ok else 'MAL'}  correcto {int(correcto):>6}   {pregunta}")

### Y ahora sin tools, para aislar al modelo

Las mismas cuentas, sin inventario de por medio. Aritmética pura.

In [ ]:
from agentlib import chat

CUENTAS = [
    ("3 * 340 + 2 * 38", 1096),
    ("7 * 340 + 13 * 62 + 11 * 85", 4121),
    ("23 * 120 + 19 * 34", 3406),
    ("1847 * 263", 485761),
]
for expr, correcto in CUENTAS:
    r = chat([{"role": "user", "content": f"¿Cuánto es {expr}? Responde solo el número."}])
    dicho = re.findall(r"\d[\d,]*", r["content"].replace(",", ""))
    n = int(dicho[0]) if dicho else None
    print(f"  {'OK ' if n == correcto else 'MAL'}  {expr:<32} = {correcto:>7}   dijo {n}")

### Lo que dicen las dos tablas de cuentas

Cuatro cosas, todas medidas el 2026-11-15 contra el servidor del Tec con
`qwen3.5:9b`.

**Falla siempre, y falla poco.** Cero de cuatro con tools, cero de cuatro sin
ellas. Y los errores son chicos: `1847 * 263` da 485 761 y el modelo dijo
487 761, que se pasa por 2 000 sobre casi medio millón. Tiene la misma cantidad
de dígitos, empieza igual, y no se detecta a ojo.

**La dificultad no predice el acierto.** Esa multiplicación de seis cifras es la
que menos se desvía de las cuatro, un 0.4%. La que más se desvía es una suma de
dos términos: `23 * 120 + 19 * 34` da 3 406, el modelo dijo 2 858, y ahí se pasa
por 548. No hay cálculos que se "pongan difíciles": hay patrones que estaban en
el entrenamiento y patrones que no.

**Es determinista.** Las cuatro cuentas sin tools dieron los mismos cuatro
números equivocados en las dos corridas completas que se hicieron ese día. Y la
pregunta de la sección anterior, corrida tres veces seguidas, contestó $1,560
las tres, con el desglose idéntico palabra por palabra.

**Y el contexto cambia la respuesta.** Esa misma pregunta contestaba $1,400 en
la corrida anterior al cambio, cuando `consultar_catalogo` todavía devolvía las
existencias, y contesta $1,560 desde que dejó de devolverlas. No cambió la
pregunta ni la temperatura: cambió lo que había en la conversación antes de
ella. Eso termina de descartar la idea de que esté calculando.

> **Un modelo de lenguaje no es una calculadora que a veces se equivoca. Es un
> generador de texto que a veces produce el número correcto.** La diferencia
> importa cuando el número va en una factura.

## La solución es una tool, y es el ejemplo canónico

La aritmética es el caso perfecto para una tool: es determinista, es verificable
y el modelo es malo en eso. Exactamente el reparto que enseña el taller.

Las dos celdas siguientes le agregan una calculadora **en caliente**, sin editar
el archivo, y vuelven a correr dos de las cuentas que falló. Las dos salen
exactas. La segunda tarda unos 7 segundos y hace doce llamadas a tools.

In [ ]:
def calcular(expresion):
    """Evalúa una expresión aritmética. Cinco líneas."""
    if not set(expresion) <= set("0123456789+-*/(). "):
        return {"error": "solo números y + - * / ( )"}
    try:
        # Lista blanca de caracteres arriba: nunca un eval() sobre texto del
        # modelo sin filtrar. Esto es un guardrail, y es el paso 4 asomandose.
        return {"expresion": expresion, "resultado": eval(expresion, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}


p2.DISPONIBLES["calcular"] = calcular
p2.HERRAMIENTAS.append({
    "type": "function",
    "function": {
        "name": "calcular",
        "description": ("Evalúa una expresión aritmética. ÚSALA SIEMPRE que tengas "
                        "que sumar, restar, multiplicar o dividir: no hagas cuentas "
                        "de cabeza."),
        "parameters": {
            "type": "object",
            "properties": {"expresion": {"type": "string",
                                         "description": "Ej: '3 * 340 + 2 * 38'"}},
            "required": ["expresion"],
        },
    },
})

print("tools disponibles ahora:", list(p2.DISPONIBLES))

In [ ]:
# Las dos cuentas que fallaron sin la tool.
p2.correr("¿Cuánto pago por 7 cafés de 1kg, 13 tés de manzanilla y 11 vasos?")
print("\n  correcto: 4121\n")
p2.correr("¿Cuánto cuesta llevarme TODO el inventario?")
print("\n  correcto: 14680")

### Lo que hizo con la tool

El caso de todo el inventario le costó **cuatro vueltas y doce llamadas**:

| vuelta | qué pidió |
|---|---|
| 1 | `consultar_catalogo({})` |
| 2 | **diez** `consultar_stock`, uno por producto, en una sola vuelta |
| 3 | `calcular`, con la expresión completa |
| 4 | nada. Respondió **$14,680**, que es el valor exacto |

La expresión que armó tiene **nueve términos**, no diez:

```
17*180 + 34*95 + 8*340 + 23*62 + 42*38 + 11*120 + 3*29 + 19*34 + 7*85
```

**Dejó fuera el `TE-200` por su cuenta.** Lo consultó en la vuelta 2, vio que
tiene 0 unidades y no lo puso en la suma. Nadie se lo pidió.

Esa vuelta 2 es también donde el bucle hace lo que antes nunca hacía: el
`for pedido in pedidos` de [`agente.py`](../src/paso2_herramienta/agente.py)
ejecuta las diez llamadas de un tirón, porque el modelo las pidió juntas.

Ahí está el reparto: **el modelo decide bien qué hay que calcular y calcula
mal.** Una tool no lo hace más capaz; le quita de encima lo que hace mal.

## Tu turno

Los cuatro `# TODO (taller)` del final de
[`src/paso2_herramienta/agente.py`](../src/paso2_herramienta/agente.py).

Después de editar el archivo, vuelve a correr la celda del `importlib.reload`:
Python no relee un módulo que ya cargó, así que sin eso seguirías corriendo la
versión vieja.

El `TODO 3` —borrar *"no inventes números"* del system prompt— es el que deja
ver mejor hasta dónde llega un prompt y dónde empieza a hacer falta código.

> **Y el `TODO 4` quedó más interesante de lo que era.** Pide bajar
> `MAX_VUELTAS` a 1 y preguntar algo que necesite dos tools. Desde que el
> catálogo no devuelve existencias, *cualquier* pregunta por cantidades lo
> necesita, así que el guardrail se dispara con la primera que se te ocurra.